# Per-Day Quality-Filtered All-Speech Poisson GLM Encoding Orchestrator

Identical to `word_level_duration_cv_filtered_speech.ipynb` but restricted to words
spoken within a single calendar day during **active hours** (default 09:00–23:00 local time).

One SLURM job is submitted per *(patient, date)* pair.  
Outputs land in `{vad_root}/{patient}/encoding/word_level_duration_cv_filtered_speech_per_day/{date}/`.

### 1. Imports

In [1]:
from pathlib import Path
import subprocess

import dill as pickle
import numpy as np
import pandas as pd

### 2. Configuration

In [2]:
# ── Paths ─────────────────────────────────────────────────────────────────────
PROJ_DIR    = Path('/mnt/labworlds/Hayden/Hayden_Lab/speech_247')
VAD_ROOT    = PROJ_DIR / 'vad_new'
WORKER_PATH = Path('/scratch/tahaismail424/speech_247/notebooks/REBIRTH_2!'
                   '/standard_encoding_analysis/poisson_glm_worker.py')
PYTHON_BIN  = '/scratch/tahaismail424/miniforge3/envs/speech_247_env/bin/python3'

PATIENTS       = None   # None => scan all Y* patient folders
FORCE_RESUBMIT = False

# ── Day-partition settings ────────────────────────────────────────────────────
LOCAL_TZ           = 'America/Chicago'   # timezone for interpreting UTC timestamps
ACTIVE_HOURS_START = 9                   # inclusive, local time
ACTIVE_HOURS_END   = 23                  # exclusive, local time
MIN_WORDS_PER_DAY  = 200                 # skip days with fewer quality-filtered words

# ── Model settings ────────────────────────────────────────────────────────────
SPIKE_OFFSET_IDX = 0
GPT2_LAYER       = -1
N_PCA            = 100
OUTER_SPLITS     = 5
INNER_SPLITS     = 5
N_ALPHAS         = 30
ALPHA_LOW        = -3.0
ALPHA_HIGH       = 3.0
N_SHUFFLES       = 50

# ── Compute mode ─────────────────────────────────────────────────────────────
USE_GPU = True

# ── SLURM ─────────────────────────────────────────────────────────────────────
CONDA_ACTIVATE = 'source /scratch/tahaismail424/.bash_profile && conda activate speech_247_env'
if USE_GPU:
    PARTITION = 'Universal'
    GRES      = 'gpu:1'
    CPUS      = 8
    MEM       = '64G'
    TIME      = '24:00:00'
else:
    PARTITION = 'Universal'
    GRES      = ''
    CPUS      = 16
    MEM       = '32G'
    TIME      = '48:00:00'

BASE_RUN_NAME      = 'word_level_duration_cv_filtered_speech_per_day'
GLOBAL_LOGS_DIR    = VAD_ROOT / f'{BASE_RUN_NAME}_slurm_logs'
GLOBAL_SCRIPTS_DIR = VAD_ROOT / f'{BASE_RUN_NAME}_slurm_scripts'
GLOBAL_LOGS_DIR.mkdir(parents=True, exist_ok=True)
GLOBAL_SCRIPTS_DIR.mkdir(parents=True, exist_ok=True)

print('vad root:   ', VAD_ROOT)
print('worker:     ', WORKER_PATH)
print('logs dir:   ', GLOBAL_LOGS_DIR)
print(f'compute:     {"GPU" if USE_GPU else "CPU"} | partition={PARTITION}')
print(f'active hours: {ACTIVE_HOURS_START}:00 – {ACTIVE_HOURS_END}:00 {LOCAL_TZ}')

vad root:    /mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_new
worker:      /scratch/tahaismail424/speech_247/notebooks/REBIRTH_2!/standard_encoding_analysis/poisson_glm_worker.py
logs dir:    /mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_new/word_level_duration_cv_filtered_speech_per_day_slurm_logs
compute:     GPU | partition=Universal
active hours: 9:00 – 23:00 America/Chicago


### 3. Compute & Save Per-Day Word Indices Per Patient

In [3]:
def compute_day_indices(patient: str) -> 'dict[str, Path]':
    """
    Load quality-filtered word df, group by local calendar date within active hours,
    and save one .npy index file per day.  Returns {date_str: path}.
    """
    patient_root = VAD_ROOT / patient
    filtered_csv = patient_root / f'{patient}_word_df_filtered.csv'

    if not filtered_csv.exists():
        print(f'  [{patient}] missing {filtered_csv.name}')
        return {}

    df = pd.read_csv(filtered_csv, usecols=['original_word_idx', 'utc_word_start'])
    df['utc_word_start'] = pd.to_datetime(df['utc_word_start'], utc=True)
    df['local_dt'] = df['utc_word_start'].dt.tz_convert(LOCAL_TZ)
    df['hour']     = df['local_dt'].dt.hour
    df['date_str'] = df['local_dt'].dt.strftime('%Y-%m-%d')

    active = df[(df['hour'] >= ACTIVE_HOURS_START) & (df['hour'] < ACTIVE_HOURS_END)]

    day_paths = {}
    for date_str, grp in active.groupby('date_str'):
        if len(grp) < MIN_WORDS_PER_DAY:
            print(f'  [{patient}] {date_str}: only {len(grp)} words — skipping')
            continue
        idx_dir  = patient_root / 'encoding' / BASE_RUN_NAME / date_str
        idx_dir.mkdir(parents=True, exist_ok=True)
        idx_path = idx_dir / f'{patient}_{date_str}_word_idx.npy'
        word_idx = grp['original_word_idx'].values.astype(np.int64)
        np.save(idx_path, word_idx)
        day_paths[date_str] = idx_path
        print(f'  [{patient}] {date_str}: {len(word_idx):,} words → {idx_path.name}')

    return day_paths


# Discover patients
if PATIENTS is None:
    all_patients = sorted([p.name for p in VAD_ROOT.iterdir() if p.is_dir() and p.name.startswith('Y')])
else:
    all_patients = PATIENTS

print(f'Computing per-day word indices for {len(all_patients)} patients...\n')
patient_day_idx_paths = {}   # {patient: {date_str: Path}}
for patient in all_patients:
    day_paths = compute_day_indices(patient)
    if day_paths:
        patient_day_idx_paths[patient] = day_paths

total_jobs = sum(len(v) for v in patient_day_idx_paths.values())
print(f'\nTotal (patient, day) pairs ready: {total_jobs}')

Computing per-day word indices for 21 patients...

  [YEY] 2024-04-01: 16,510 words → YEY_2024-04-01_word_idx.npy
  [YEY] 2024-04-02: 23,841 words → YEY_2024-04-02_word_idx.npy
  [YEZ] 2024-04-09: 6,698 words → YEZ_2024-04-09_word_idx.npy
  [YEZ] 2024-04-10: 16,464 words → YEZ_2024-04-10_word_idx.npy
  [YEZ] 2024-04-11: 1,584 words → YEZ_2024-04-11_word_idx.npy
  [YEZ] 2024-04-12: 7,996 words → YEZ_2024-04-12_word_idx.npy
  [YEZ] 2024-04-14: 18,298 words → YEZ_2024-04-14_word_idx.npy
  [YEZ] 2024-04-15: 14,595 words → YEZ_2024-04-15_word_idx.npy
  [YFA] 2024-04-23: 18,729 words → YFA_2024-04-23_word_idx.npy
  [YFA] 2024-04-24: 58,808 words → YFA_2024-04-24_word_idx.npy
  [YFA] 2024-04-25: 48,473 words → YFA_2024-04-25_word_idx.npy
  [YFA] 2024-04-26: 36,215 words → YFA_2024-04-26_word_idx.npy
  [YFA] 2024-04-27: 66,950 words → YFA_2024-04-27_word_idx.npy
  [YFA] 2024-04-28: 43,939 words → YFA_2024-04-28_word_idx.npy
  [YFA] 2024-04-29: 54,695 words → YFA_2024-04-29_word_idx.npy
  [YFA]

### 4. Resolve Input Paths & Build Status Table

In [5]:
def first_existing(paths):
    for path in paths:
        if path is not None and Path(path).exists():
            return Path(path)
    return None


def resolve_patient_base_paths(patient: str) -> dict:
    patient_root = VAD_ROOT / patient
    embeddings_path = first_existing([
        patient_root / 'embeddings' / f'{patient}_gpt2_embeddings.npy',
        patient_root / 'all_convo_recording' / 'all_words_filtered_all_layers_gpt2.npy',
    ])
    counts_path = first_existing([
        patient_root / 'neural_embeddings' / 'word_spike_counts_offsets_all.npy',
        patient_root / 'all_convo_recording' / 'word_spike_counts_offsets_all.npy',
    ])
    durations_path = first_existing([
        patient_root / 'neural_embeddings' / 'word_durs.npy',
        patient_root / 'all_convo_recording' / 'word_durs.npy',
    ])
    return {
        'patient_root': patient_root,
        'embeddings_path': embeddings_path,
        'counts_path': counts_path,
        'durations_path': durations_path,
    }


def build_status_df() -> pd.DataFrame:
    rows = []
    for patient in all_patients:
        base = resolve_patient_base_paths(patient)
        day_idx_map = patient_day_idx_paths.get(patient, {})
        for date_str, idx_path in day_idx_map.items():
            out_dir      = base['patient_root'] / 'encoding' / BASE_RUN_NAME / date_str
            result_pkl   = out_dir / f'{patient}_encoding_results_cv.pkl'
            result_tar   = out_dir / f'{patient}_encoding_models_cv.tar'
            meta_json    = out_dir / f'{patient}_meta.json'
            success_path = out_dir / f'{patient}_SUCCESS'
            error_path   = out_dir / f'{patient}_error.txt'

            has_emb  = base['embeddings_path'] is not None
            has_cnt  = base['counts_path'] is not None
            has_dur  = base['durations_path'] is not None
            ready    = has_emb and has_cnt and has_dur
            success  = success_path.exists()

            rows.append({
                'patient':        patient,
                'date':           date_str,
                'patient_root':   base['patient_root'],
                'embeddings_path': base['embeddings_path'],
                'counts_path':    base['counts_path'],
                'durations_path': base['durations_path'],
                'word_idx_path':  idx_path,
                'out_dir':        out_dir,
                'result_pkl':     result_pkl,
                'result_tar':     result_tar,
                'meta_json':      meta_json,
                'success_path':   success_path,
                'error_path':     error_path,
                'has_embeddings': has_emb,
                'has_counts':     has_cnt,
                'has_durations':  has_dur,
                'ready_inputs':   ready,
                'has_success':    success,
                'has_error':      error_path.exists(),
                'has_result_pkl': result_pkl.exists(),
                'needs_submit':   ready and (FORCE_RESUBMIT or not success),
            })
    return pd.DataFrame(rows).sort_values(['patient', 'date']).reset_index(drop=True)


status_df = build_status_df()
print(f'(patient, day) pairs total:  {len(status_df)}')
print(f'ready inputs:                {int(status_df["ready_inputs"].sum())}')
print(f'completed:                   {int(status_df["has_success"].sum())}')
print(f'errors:                      {int(status_df["has_error"].sum())}')
print(f'needs submit:                {int(status_df["needs_submit"].sum())}')
status_df[['patient', 'date', 'has_embeddings', 'has_counts', 'has_durations',
           'ready_inputs', 'has_success', 'has_error', 'needs_submit']]

(patient, day) pairs total:  156
ready inputs:                156
completed:                   156
errors:                      0
needs submit:                0


,patient,date,has_embeddings,has_counts,has_durations,ready_inputs,has_success,has_error,needs_submit
0,YEY,2024-04-01,True,True,True,True,True,False,False
1,YEY,2024-04-02,True,True,True,True,True,False,False
2,YEZ,2024-04-09,True,True,True,True,True,False,False
3,YEZ,2024-04-10,True,True,True,True,True,False,False
4,YEZ,2024-04-11,True,True,True,True,True,False,False
...,...,...,...,...,...,...,...,...,...
151,YFU,2025-12-18,True,True,True,True,True,False,False
152,YFV,2026-02-17,True,True,True,True,True,False,False
153,YFV,2026-02-18,True,True,True,True,True,False,False
154,YFV,2026-02-19,True,True,True,True,True,False,False


### 5. Submit SLURM Jobs

In [6]:
submitted = []
failed    = []

for _, row in status_df.iterrows():
    patient  = row['patient']
    date_str = row['date']
    if not row['needs_submit']:
        continue

    row['out_dir'].mkdir(parents=True, exist_ok=True)
    reset_line = (
        f"rm -f {row['success_path']} {row['error_path']}" if FORCE_RESUBMIT else 'true'
    )

    cmd = (
        f'{PYTHON_BIN} {WORKER_PATH} {patient} {VAD_ROOT} {row["out_dir"]} '
        f'--spike-offset-idx {SPIKE_OFFSET_IDX} '
        f'--gpt2-layer {GPT2_LAYER} '
        f'--n-pca {N_PCA} '
        f'--outer-splits {OUTER_SPLITS} '
        f'--inner-splits {INNER_SPLITS} '
        f'--n-alphas {N_ALPHAS} '
        f'--alpha-low {ALPHA_LOW} '
        f'--alpha-high {ALPHA_HIGH} '
        f'--n-shuffles {N_SHUFFLES} '
        f'--embeddings-path {row["embeddings_path"]} '
        f'--counts-path {row["counts_path"]} '
        f'--durations-path {row["durations_path"]} '
        f'--word-idx-path {row["word_idx_path"]}'
    )

    job_name = f'glm_pd_{patient}_{date_str}'
    gres_lines = [f'#SBATCH --gres={GRES}'] if GRES else []
    lines = [
        '#!/bin/bash',
        f'#SBATCH --job-name={job_name}',
        f'#SBATCH --partition={PARTITION}',
        *gres_lines,
        f'#SBATCH --cpus-per-task={CPUS}',
        f'#SBATCH --qos=default_tier',
        f'#SBATCH --exclude=guppy-1',
        f'#SBATCH --mem={MEM}',
        f'#SBATCH --time={TIME}',
        f'#SBATCH --output={GLOBAL_LOGS_DIR}/{patient}_{date_str}_%j.out',
        f'#SBATCH --error={GLOBAL_LOGS_DIR}/{patient}_{date_str}_%j.err',
        '',
        'set -eo pipefail',
        '',
        CONDA_ACTIVATE,
        '',
        'echo "HOSTNAME: $(hostname)"',
        'echo "START: $(date)"',
        f'echo "PATIENT: {patient}"',
        f'echo "DATE: {date_str}"',
        '',
        reset_line,
        '',
        cmd,
        '',
        'echo "END: $(date)"',
    ]

    sbatch_path = GLOBAL_SCRIPTS_DIR / f'{patient}_{date_str}.sbatch'
    sbatch_path.write_text('\n'.join(lines))

    try:
        res = subprocess.run(['sbatch', str(sbatch_path)], capture_output=True, text=True, check=True)
        submitted.append((patient, date_str, res.stdout.strip()))
        print(f'submitted: {patient} {date_str} -> {res.stdout.strip()}')
    except subprocess.CalledProcessError as exc:
        failed.append((patient, date_str, exc.stderr.strip()))
        print(f'FAILED SUBMISSION: {patient} {date_str}\n{exc.stderr}')

print(f'submitted={len(submitted)}, failed={len(failed)}')

submitted=0, failed=0


### 6. Check Status

In [7]:
status_df = build_status_df()
print(f'completed: {int(status_df["has_success"].sum())} / {len(status_df)}')
status_df[['patient', 'date', 'ready_inputs', 'has_success', 'has_error', 'has_result_pkl']]

completed: 156 / 156


,patient,date,ready_inputs,has_success,has_error,has_result_pkl
0,YEY,2024-04-01,True,True,False,True
1,YEY,2024-04-02,True,True,False,True
2,YEZ,2024-04-09,True,True,False,True
3,YEZ,2024-04-10,True,True,False,True
4,YEZ,2024-04-11,True,True,False,True
...,...,...,...,...,...,...
151,YFU,2025-12-18,True,True,False,True
152,YFV,2026-02-17,True,True,False,True
153,YFV,2026-02-18,True,True,False,True
154,YFV,2026-02-19,True,True,False,True


### 7. Inspect Errors

In [8]:
error_rows = status_df[status_df['has_error']].copy()
print(f'(patient, day) pairs with error files: {len(error_rows)}')
for _, row in error_rows.head(10).iterrows():
    print('=' * 100)
    print(row['patient'], row['date'])
    print(row['error_path'].read_text()[:4000])

(patient, day) pairs with error files: 0


### 8. Load & Summarise Results

In [9]:
records = []

for _, row in status_df.iterrows():
    if not row['has_result_pkl']:
        continue
    with open(row['result_pkl'], 'rb') as f:
        df = pickle.load(f)
    if 'is_summary' not in df.columns:
        continue
    summary_df = df[df['is_summary'] == True].copy()
    if summary_df.empty:
        continue
    summary_df['patient'] = row['patient']
    summary_df['date']    = row['date']
    records.append(summary_df)

if records:
    all_summary = pd.concat(records, ignore_index=True)
    print('loaded (patient, day) pairs:', len(records))
    print('summary rows:', len(all_summary))
    display(all_summary.head())

    day_level = all_summary.groupby(['patient', 'date']).agg(
        n_neurons=('neuron_idx', 'nunique'),
        pseudo_r2_mean=('pseudo_r2_mean', 'mean'),
        pearson_corr_mean=('pearson_corr_mean', 'mean'),
        spearman_corr_mean=('spearman_corr_mean', 'mean'),
    ).reset_index()
    display(day_level.sort_values(['patient', 'date']))
else:
    print('No completed results found yet.')

loaded (patient, day) pairs: 156
summary rows: 8280


,neuron_idx,fold_id,is_summary,outer_splits,inner_splits,best_alpha,alpha_ll_mean,alpha_ll_std,ll_real,ll_null,...,spearman_corr_mean,spearman_corr_std,edf_mean,edf_std,aic_mean,aic_std,bic_mean,bic_std,patient,date
0,0,NaN,True,5,5,NaN,NaN,NaN,NaN,NaN,...,0.661159,0.007204,98.458432,1.698551,24302.032813,690.710880,24902.854157,692.075492,YEY,2024-04-01
1,1,NaN,True,5,5,NaN,NaN,NaN,NaN,NaN,...,0.658946,0.008385,96.714920,1.045148,24491.958984,742.307870,25082.141010,737.738582,YEY,2024-04-01
2,2,NaN,True,5,5,NaN,NaN,NaN,NaN,NaN,...,0.668030,0.006773,98.206139,0.746537,23769.107812,794.278136,24368.389692,796.212928,YEY,2024-04-01
3,3,NaN,True,5,5,NaN,NaN,NaN,NaN,NaN,...,0.664416,0.005452,98.751401,1.385215,24366.278906,595.834769,24968.887638,592.091526,YEY,2024-04-01
4,4,NaN,True,5,5,NaN,NaN,NaN,NaN,NaN,...,0.666762,0.006569,97.549745,1.693846,23820.909766,707.100765,24416.185859,703.587539,YEY,2024-04-01


,patient,date,n_neurons,pseudo_r2_mean,pearson_corr_mean,spearman_corr_mean
0,YEY,2024-04-01,16,0.201326,0.633800,0.657270
1,YEY,2024-04-02,16,0.064652,0.697249,0.629217
2,YEZ,2024-04-09,24,0.044805,0.570063,0.517298
3,YEZ,2024-04-10,24,0.095386,0.486519,0.510818
4,YEZ,2024-04-11,24,-0.017925,0.487027,0.416693
...,...,...,...,...,...,...
151,YFU,2025-12-18,64,0.074328,0.620833,0.590089
152,YFV,2026-02-17,96,0.012348,0.745009,0.556844
153,YFV,2026-02-18,96,0.045287,0.680817,0.543811
154,YFV,2026-02-19,96,0.006112,0.789431,0.563089
